In [1]:
import warnings
warnings.filterwarnings('ignore')

# for interactive 
from itables import init_notebook_mode
init_notebook_mode(all_interactive = True)
import itables.options as opt
opt.maxBytes = 2**20
opt.maxColumns= 0

import pandas as pd 
#pd.options.display.float_format = '{:.2%}'.format

import numpy as np
from os import path 

import pyfolio as pf

import ffn

import pypfopt
from pypfopt import plotting
from pypfopt.expected_returns import *


import empyrical 
from datetime import date, timedelta

import riskfolio as rp


<IPython.core.display.Javascript object>

In [2]:
cd /Users/safishajjouz/GitHub/myPythonPackages

/Users/safishajjouz/GitHub/myPythonPackages


In [3]:
from myPortfolioManagement.myData import * 
from myPortfolioManagement.myPortfolioSelection import *
from myPortfolioManagement.myPortfolioOptimisation import *
from myPortfolioManagement.myPerformanceAnalytics import *
from myPortfolioManagement.myPlots import *

In [4]:
dataframe = load_fidelity_prices(filter_date = '2006-01-01')
dataframe = dataframe[dataframe['Asset_Class'].isin(['Equity', 'Alternatives'])]
dataframe = dataframe[dataframe.index<='2021-12-01']

df_overview = mydata_overview(dataframe, 
                my_assets_col_name = 'fund',
                my_date_col_name = 'Date', 
                price_col_name = 'price')

#dataframe = dataframe[dataframe.index<=min(df_overview.Date_max)]
#list_of_funds  = df_overview[df_overview['years_available']>= 5]
#list_of_funds = list_of_funds['fund'].to_list()
#dataframe = dataframe[dataframe['fund'].isin(list_of_funds)]
#qgrid.show_grid(dataframe)

In [7]:
df_overview.set_index('fund')

Loading... (need help?)


In [8]:
# from long to wide 
df = dataframe.pivot_table(index='Date', 
                        columns='fund', 
                        values='price')

In [9]:
# benchmark
bench = ['Vanguard FTSE Dev Wld ex-UK Eq Idx £ Acc', 
         'iShares Core S&P 500 ETF USD Acc GBP',
        'Invesco EQQQ NASDAQ-100 ETF GBP'][1]


### Returns 

In [10]:
# log returns 
ret_log = returns_from_prices(df, log_returns = True)

#simple returns 
ret_simple = returns_from_prices(df)

# calculate rolling mean 
df_rolling = df.resample('Y').mean().pct_change().rolling(3).mean()


## Performance Statistics 

In [15]:
basics = ['start', 'end', 'total_return', 'cagr', 'yearly_mean','yearly_vol', 'max_drawdown']
look_back_ret = ['mtd', 'three_month', 'six_month', 'ytd', 'one_year', 'three_year',
                 'five_year', 'ten_year', 'incep']
ratios =  ['calmar','yearly_sharpe', 'yearly_sortino'] 
rest =    ['best_year', 'worst_year','win_year_perc']
df_stats = performance_overview(ret_simple, prices = False)
df_stats[basics]

Loading... (need help?)


### Expected  Returns 

In [17]:
# calculate rolling mean 
df_yearly_returns = df.resample('Y').mean().pct_change()
df_yearly_returns.mean()

Loading... (need help?)


In [18]:
mean_historical_return(df_yearly_returns, 
                       returns_data=True, 
                       compounding=False, frequency=1)

Loading... (need help?)


In [19]:
ema_historical_return(df_yearly_returns, 
                      returns_data=True, 
                      compounding=True, span=3, 
                      frequency=1)

Loading... (need help?)


In [20]:
ret_capm = df_yearly_returns

capm_return(ret_capm.drop(bench, axis = 1), 
            market_prices=ret_capm[[bench]],
                      returns_data=True, 
                      compounding=True,
                      risk_free_rate=0.02,
                      frequency=1)

Loading... (need help?)


### Factor Attributes

In [21]:
df_greeks = alpha_beta(df_rolling, 
                       benchmark = 'iShares Core S&P 500 ETF USD Acc GBP', 
                       my_date_col_name = 'Date', 
                       returns_rolling = True)

In [22]:
df_greeks

Loading... (need help?)


In [ ]:
ffn.core.deannualize(df_rolling, nperiods = 252)

In [23]:
information_ratio(df_rolling , benchmark = bench, my_assets_col_name = 'fund')

Loading... (need help?)


### Coveriance Matrix Based on historical Returns 

In [30]:
dct = ret_simple.calc_ftca(threshold=0.7)  # higher values more clusters 

df_cluster = pd.DataFrame.from_dict(dct, orient='index')
df_cluster = (pd.DataFrame.from_dict(dct, orient='index').T
   .melt(var_name='cluster', value_name='fund')
   .dropna(subset=['fund']))
df_cluster['cluster'] = [f"{label}" for label in df_cluster['cluster']]
df_cluster

Loading... (need help?)


In [ ]:
myassets = ['Scottish Mortgage Ord', 
            'Invesco EQQQ NASDAQ-100 ETF GBP',
            'Fundsmith Equity I Acc', 
            'HarbourVest Global Priv Equity Ord',
            'JPM Global Macro Opportunities C Net Acc']

benchmark = ['Vanguard FTSE Dev Wld ex-UK Eq Idx £ Acc'] 



In [ ]:

df_myportfolio = culculate_portfolio_returns(ret_port.dropna(), 
                                myassets_list = ret_port.columns.to_list(),
                                myweights_list = [1/3]*5,
                                portfolio_name = 'myPortfolio')
df_myportfolio.resample('Y').mean().pct_change().rolling(3).mean()

df_rolling = df_rolling.drop(myassets, axis = 1).merge(df_myportfolio, on = 'Date')

df_rolling

In [ ]:
dataframe = load_fidelity_prices(filter_date = '2010-01-01')
dataframe = dataframe[dataframe['Asset_Class'] != 'Equity']

df_overview = mydata_overview(dataframe, 
                my_assets_col_name = 'fund',
                my_date_col_name = 'Date')

#dataframe = dataframe[dataframe.index<=min(df_overview.Date_max)]
#list_of_funds  = df_overview[df_overview['years_available']>= 5]
#list_of_funds = list_of_funds['fund'].to_list()
#dataframe = dataframe[dataframe['fund'].isin(list_of_funds)]

# from long to wide 
df = dataframe.pivot_table(index='Date', 
                        columns='fund', 
                        values='price')
df_rolling2 = df.resample('Y').mean().pct_change().rolling(3).mean()
df_rolling2 = df_rolling2.merge(df_rolling, on = 'Date')
df_rolling2 = df_rolling2.drop(df_rolling.columns.to_list(), axis = 1)
df_rolling2 = df_rolling2.merge(df_myportfolio, on = 'Date')
df_rolling2

In [ ]:
df_corr = df_rolling2.corr()
#df_corr = df_corr[df_corr<0.5]
plt.figure(figsize=(18, 10))
heatmap = sns.heatmap(df_corr, vmin=-1, vmax=1, annot=True, cmap='BrBG')
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':18}, pad=12);

In [ ]:
# download from Yahoo 
# Set dates 
start_date = '2010-01-01'
end_date = date.today() -  timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d") 

df = get_stock_prices(yahoo_tickers = ['^GSPC'], 
                 start_date = start_date,
                 end_date = end_date, 
                 time_interval = 'daily', 
                 num_cpus = 5) 

In [ ]:
df.head()

In [ ]:
# from long to wide 
df = df.pivot_table(index=['Date'], 
                        columns='stock', 
                        values='adjclose')
    


In [ ]:
# calculate rolling mean 
df_rolling = df.resample('Y').mean().pct_change().rolling(1).mean()

### Read Data From Fidelity Files

In [ ]:
# import list of funds 

dataframe = load_fidelity_prices()

In [ ]:
dataframe = dataframe[['Date', 'fund', 'price']]

In [ ]:
# import list of funds 
mypath    = '/Users/safishajjouz/GitHub/myPortfolioManagement/files'
file_name = "list_of_funds.xlsx"
file_path_to_load = path.join(mypath, file_name)

# load prices 
df = pd.read_excel(file_path_to_load)
df = df[df.my_portfolio == 'T']
df = df.drop(['Trust', 'ETF','my_portfolio', 'market_neutral'], axis =1)

index_symbols = ['^IXIC', # Nasdaq 
                 '^DJI',    # Dow Jones 
                 '^GSPC']   # S&P Index 
mylist = index_symbols + df.dropna(subset=['yahoo_ticker']).yahoo_ticker.to_list()

### Load Data From Yahoo 

In [ ]:
# Set dates 
start_date = '1985-01-01'
end_date = date.today() -  timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d") 

# Load Index 
index_symbols = ['^IXIC', # Nasdaq 
                 '^DJI',    # Dow Jones 
                 '^GSPC']   # S&P Index 

# load my list of funds 
mylist = index_symbols + df.dropna(subset=['yahoo_ticker']).yahoo_ticker.to_list()

# download from Yahoo 
df = get_stock_prices(yahoo_tickers =  mylist, 
                 start_date = start_date,
                 end_date = end_date, 
                 time_interval = 'daily', 
                 num_cpus = 5) 
df = df[['close', 'volume', 'stock']]

In [ ]:
mydata_overview(dataframe, my_assets_col_name = 'stock', my_date_col_name = 'Date').sort_values('years_available', ascending = False)

In [ ]:
df_mutual_fun_prices = get_mutual_funds_prices(mutual_fund_isin =  ['GB0006063233'], 
                                               start_date = start_date,
                                               end_date = end_date, 
                                               num_cpus = 5)

In [ ]:
# constract Reward Metric 



In [ ]:
# Construct Consistency Metric 
df_cleaned[[col_name]].resample('M').mean().pct_change().dropna()

### Clustering 

In [31]:
# convert from Long to wide dataframe 
df_cleaned = df_index.pivot_table(index=["Date"], 
                    columns='stock', 
                    values='close')

df_cleaned

NameError: name 'df_index' is not defined

In [ ]:
from sklearn.preprocessing import normalize

smoothing_tech = ['ML', 'Kalman', 'Moving_Average', 'Exponential_smoothing']

In [ ]:
def ml_smoothing(df, smoothing_parameter = 0.05):
    
    df_cleaned = df_cleaned.dropna()
    df_cleaned.index = pd.to_datetime(df_cleaned.index, format='%Y/%m/%d')
    df_cleaned.interpolate(method='time', inplace=True)
    
    # smooth Series 
    data_series = df_cleaned.transpose().to_numpy()
    df_cleaned_smoothed = LowessSmoother(smooth_fraction=smoothing_parameter)  # higher values for smooth fraction more smooth (trend)
    df_cleaned_smoothed.smooth(data_series)
    
    # generate smoothed DataFrame 
    df_cleaned_sm = df_cleaned.copy()
    for i in range(len(df_cleaned.columns)):
        df_cleaned_sm.iloc[:,i] = df_cleaned_smoothed.smooth_data[i]
    return df_cleaned_sm

In [ ]:
def smooth_series(df, smoothing_technique = 'ML', 
                          smoothing_parameter = 0.05, 
                 smooth_fraction = smooth_fraction):
    
    if 
    # to smooth series 
    
        
       

In [ ]:
X = df_cleaned_sm[['Dow Jones Industrial Average']] # you options here are: inflation_core_trend and inflation_trend
X_scaled = normalize(X, axis=0)

df_scores = []
k_values_to_try = np.arange(2, 10)
Sum_of_squared_distances = []
for n_clusters in k_values_to_try:
    #Perform clustering.
    kmeans = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", max_iter=10, random_state=33)
    labels_clusters = kmeans.fit(X_scaled)
    Sum_of_squared_distances.append(kmeans.inertia_)

plt.plot(k_values_to_try,Sum_of_squared_distances,'bx-')
plt.xlabel('Values of K') 
plt.ylabel('Sum of squared distances/Inertia') 
plt.title('Elbow Method For Optimal k')
plt.show()

# after optimal clustering 
optimal_clusters = 5
smooth_km = TimeSeriesKMeans(n_clusters=optimal_clusters, metric="dtw", max_iter=10, random_state=33)
smooth_km.fit(X_scaled)

# present reslults 
labels = smooth_km.labels_
fancy_names_for_labels = labels + 1   #[f"Cluster {label+1}" for label in labels]
df_tsclusters = pd.DataFrame(zip(df_cleaned_sm.transpose(), fancy_names_for_labels),
                           columns=["year","Regime"]).sort_values(by="year").set_index("year")
df_tsclusters.head()

df_cleaned_sm['Regime_TS'] = list(smooth_km.labels_ + 1)

df_cleaned_sm

In [ ]:
df_cleaned_sm.reset_index().plot.scatter(x="Date", y='Dow Jones Industrial Average', 
                          c="Regime_TS", 
                          cmap="viridis", figsize = (16,10), sharex=False);

                                                             
plt.ylabel('Nasdaq', fontsize=20)
plt.xlabel('Year', fontsize=20)
plt.legend(fontsize=15)
plt.title('Nasdaq Regimes',
          fontsize = 20)

In [ ]:
def plot_box_lots(df, col_name = None):
     
    df_sp_monthly = df_cleaned[[col_name]].resample('M').mean().pct_change().dropna()
    df_sp_monthly = df_sp_monthly[df_sp_monthly.index>='1990-01-01']
    df_sp_monthly['year'] = pd.DatetimeIndex(df_sp_monthly.index).year
    
    fig, ax = plt.subplots(figsize=(25,15))
    plt.title(col_name + ': Distribution of Returns over time', fontdict={'fontsize':20})
    plt.xlabel('Year', fontsize=18)
    plt.ylabel('Price', fontsize=16)
    seaborn.boxplot(df_sp_monthly['year'], df_sp_monthly[col_name], ax=ax)
    plt.locator_params(axis="x", nbins=20)
    ax.xaxis.set_tick_params(labelsize=20)
    plt.xticks(rotation=45)
    plt.axhline(y = 0.01038, color = 'r', linestyle = '-')

In [ ]:
plot_box_lots(df_cleaned, col_name = 'NASDAQ Composite')

In [ ]:
df_cleaned[['NASDAQ Composite']].resample('M').mean().pct_change().dropna().mean()

In [ ]:
factors = ['MTUM', 'QUAL', 'VLUE', 'SIZE', 'USMV']
# download from Yahoo 
# Set dates 
start_date = '1995-01-01'
end_date = date.today() -  timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d") 

df_factors = get_stock_prices(yahoo_tickers = factors, 
                 start_date = start_date,
                 end_date = end_date, 
                 time_interval = 'daily', 
                 num_cpus = 5)

mydata_overview(df_factors, 
                my_assets_col_name = 'stock',
                my_date_col_name = 'Date', 
                price_col_name = 'adjclose')

X = df_factors.pivot_table(index='Date', 
                        columns='stock', 
                        values='adjclose')

# Calculating returns

X = X.pct_change().dropna()

ret_simple = ret_simple.merge(X, on = 'Date')

step = 'Forward' # Could be Forward or Backward stepwise regression
loadings = rp.loadings_matrix(X=ret_simple[X.columns.to_list()], 
                              Y=ret_simple.drop(X.columns.to_list(), axis = 1), 
                              stepwise=step)

loadings.style.format("{:.4f}").background_gradient(cmap='RdYlGn')

In [ ]:

fig, ax = plt.subplots(figsize=(25,15))
plt.title('Distribution of Returns over time', fontdict={'fontsize':20})
plt.xlabel('Year', fontsize=18)
plt.ylabel('Price', fontsize=16)
seaborn.boxplot(df_sp_monthly['year'], df_sp_monthly['Dow Jones Industrial Average'], ax=ax)
plt.locator_params(axis="x", nbins=20)
ax.xaxis.set_tick_params(labelsize=20)
plt.xticks(rotation=45)